# 01 — Python Execution Model & CPython Internals
Run each cell with **Shift+Enter**

## 1. Inspecting the AST

In [ ]:
import ast

source = '''
x = 1 + 2
if x > 2:
    print(x)
'''

tree = ast.parse(source)
print(ast.dump(tree, indent=2))

## 2. Disassembling Bytecode

In [ ]:
import dis

def add(a, b):
    return a + b

print('=== add(a, b) ===')
dis.dis(add)

def conditional(x):
    if x > 0:
        return 'positive'
    return 'non-positive'

print('\n=== conditional(x) ===')
dis.dis(conditional)

In [ ]:
# Inspect code object attributes
def example(x, y=10, *args, **kwargs):
    z = x + y
    return z

code = example.__code__
print(f'co_name:      {code.co_name}')
print(f'co_varnames:  {code.co_varnames}')
print(f'co_consts:    {code.co_consts}')
print(f'co_argcount:  {code.co_argcount}')
print(f'co_flags:     {code.co_flags}')
print(f'co_stacksize: {code.co_stacksize}')

## 3. Reference Counting

In [ ]:
import sys

x = [1, 2, 3]
print(f'After creation:  refcount = {sys.getrefcount(x) - 1}')  # -1 for getrefcount arg

y = x
print(f'After y = x:     refcount = {sys.getrefcount(x) - 1}')

z = [x, x]
print(f'After z = [x,x]: refcount = {sys.getrefcount(x) - 1}')

del y
del z
print(f'After del y, z:  refcount = {sys.getrefcount(x) - 1}')

## 4. Integer Caching (-5 to 256)

In [ ]:
# CPython caches small integers
for n in [0, 100, 256, 257, 1000]:
    a = n
    b = n
    print(f'{n:5}: a is b = {a is b}, id(a)={id(a)}, id(b)={id(b)}')

## 5. Closures and Free Variables

In [ ]:
def outer(x):
    def inner(y):
        return x + y   # x is a free variable
    return inner

add5 = outer(5)
print(add5(3))   # 8

# Inspect closure
print(f'Free vars: {add5.__code__.co_freevars}')  # ('x',)
print(f'Closure:   {add5.__closure__}')            # cell objects
print(f'Cell value: {add5.__closure__[0].cell_contents}')  # 5

## 6. sys.modules Cache

In [ ]:
import sys
import os
import json

# Check what's already imported
print('os in sys.modules:', 'os' in sys.modules)
print('json in sys.modules:', 'json' in sys.modules)

# Import is cached — second import is instant
import timeit
t1 = timeit.timeit('import os', number=10000)
print(f'import os (cached): {t1*1000:.3f}ms for 10000 calls')

## 7. Garbage Collection

In [ ]:
import gc

# Create a reference cycle
class Node:
    def __init__(self, val):
        self.val = val
        self.next = None
    def __del__(self):
        print(f'Node({self.val}) deleted')

# Cycle: a -> b -> a
a = Node(1)
b = Node(2)
a.next = b
b.next = a

del a, b  # refcount doesn't reach 0 due to cycle
print('Before gc.collect()')
collected = gc.collect()  # cyclic GC cleans it up
print(f'gc.collect() freed {collected} objects')

## 8. Performance: Bytecode Optimization

In [ ]:
import dis

# Constant folding — Python optimizes at compile time
def with_constants():
    return 2 * 3 * 4  # compiled to 24 directly

def with_variables(x):
    return x * 3 * 4  # cannot fold

print('=== with_constants ===')
dis.dis(with_constants)
print('\n=== with_variables ===')
dis.dis(with_variables)

## 9. Frame Inspection

In [ ]:
import inspect

def level3():
    frame = inspect.currentframe()
    stack = inspect.stack()
    print('Call stack:')
    for info in stack:
        print(f'  {info.function} @ line {info.lineno}')

def level2():
    level3()

def level1():
    level2()

level1()

## 10. GIL Demo: CPU vs I/O Bound

In [ ]:
import threading
import time

def cpu_task(n=5_000_000):
    total = 0
    for i in range(n):
        total += i
    return total

def io_task():
    time.sleep(0.5)

# Sequential CPU
start = time.perf_counter()
cpu_task(); cpu_task()
seq_cpu = time.perf_counter() - start

# Threaded CPU (GIL prevents speedup)
t1 = threading.Thread(target=cpu_task)
t2 = threading.Thread(target=cpu_task)
start = time.perf_counter()
t1.start(); t2.start()
t1.join(); t2.join()
thread_cpu = time.perf_counter() - start

# Threaded I/O (GIL released during sleep)
t1 = threading.Thread(target=io_task)
t2 = threading.Thread(target=io_task)
start = time.perf_counter()
t1.start(); t2.start()
t1.join(); t2.join()
thread_io = time.perf_counter() - start

print(f'Sequential CPU:  {seq_cpu:.3f}s')
print(f'Threaded CPU:    {thread_cpu:.3f}s  (no speedup — GIL)')
print(f'Threaded I/O:    {thread_io:.3f}s  (~0.5s — GIL released)')